# mtcars Linear Regression Agents

Three-agent workflow showing how a modelling agent fits a linear regression on the `mtcars` dataset (target: `mpg`),
an evaluation agent runs diagnostics with LLM assistance, and a recommendation agent leverages a domain knowledge
base plus diagnostics to suggest improvements. A reinforcement-learning trainer now tunes the ridge penalty across
iterations, publishing its adjustments via the shared message bus.

In [1]:
# Cell 1: imports, dataset, knowledge bases, and shared utilities

import os
from typing import Mapping, Sequence
from getpass import getpass

from agentic_codex import AgentBuilder, Context, FunctionAdapter
from agentic_codex.core.capabilities import ContextCapability, MessageBusCapability, ResourceCapability
from agentic_codex.core.schemas import AgentStep, Message
from agentic_codex.core.message_bus import MessageBus
from agentic_codex.core.llm import EnvOpenAIAdapter

# Rooted mtcars dataset with key attributes for the regression demo
MTCARS_DATA = [
    {"name": "Mazda RX4", "mpg": 21.0, "cyl": 6, "disp": 160.0, "hp": 110, "drat": 3.9, "wt": 2.62, "qsec": 16.46, "vs": 0, "am": 1, "gear": 4, "carb": 4},
    {"name": "Mazda RX4 Wag", "mpg": 21.0, "cyl": 6, "disp": 160.0, "hp": 110, "drat": 3.9, "wt": 2.875, "qsec": 17.02, "vs": 0, "am": 1, "gear": 4, "carb": 4},
    {"name": "Datsun 710", "mpg": 22.8, "cyl": 4, "disp": 108.0, "hp": 93, "drat": 3.85, "wt": 2.32, "qsec": 18.61, "vs": 1, "am": 1, "gear": 4, "carb": 1},
    {"name": "Hornet 4 Drive", "mpg": 21.4, "cyl": 6, "disp": 258.0, "hp": 110, "drat": 3.08, "wt": 3.215, "qsec": 19.44, "vs": 1, "am": 0, "gear": 3, "carb": 1},
    {"name": "Hornet Sportabout", "mpg": 18.7, "cyl": 8, "disp": 360.0, "hp": 175, "drat": 3.15, "wt": 3.44, "qsec": 17.02, "vs": 0, "am": 0, "gear": 3, "carb": 2},
    {"name": "Valiant", "mpg": 18.1, "cyl": 6, "disp": 225.0, "hp": 105, "drat": 2.76, "wt": 3.46, "qsec": 20.22, "vs": 1, "am": 0, "gear": 3, "carb": 1},
    {"name": "Duster 360", "mpg": 14.3, "cyl": 8, "disp": 360.0, "hp": 245, "drat": 3.21, "wt": 3.57, "qsec": 15.84, "vs": 0, "am": 0, "gear": 3, "carb": 4},
    {"name": "Merc 240D", "mpg": 24.4, "cyl": 4, "disp": 146.7, "hp": 62, "drat": 3.69, "wt": 3.19, "qsec": 20.0, "vs": 1, "am": 0, "gear": 4, "carb": 2},
    {"name": "Merc 230", "mpg": 22.8, "cyl": 4, "disp": 140.8, "hp": 95, "drat": 3.92, "wt": 3.15, "qsec": 22.9, "vs": 1, "am": 0, "gear": 4, "carb": 2},
    {"name": "Merc 280", "mpg": 19.2, "cyl": 6, "disp": 167.6, "hp": 123, "drat": 3.92, "wt": 3.44, "qsec": 18.3, "vs": 1, "am": 0, "gear": 4, "carb": 4},
    {"name": "Merc 280C", "mpg": 17.8, "cyl": 6, "disp": 167.6, "hp": 123, "drat": 3.92, "wt": 3.44, "qsec": 18.9, "vs": 1, "am": 0, "gear": 4, "carb": 4},
    {"name": "Merc 450SE", "mpg": 16.4, "cyl": 8, "disp": 275.8, "hp": 180, "drat": 3.07, "wt": 4.07, "qsec": 17.4, "vs": 0, "am": 0, "gear": 3, "carb": 3},
    {"name": "Merc 450SL", "mpg": 17.3, "cyl": 8, "disp": 275.8, "hp": 180, "drat": 3.07, "wt": 3.73, "qsec": 17.6, "vs": 0, "am": 0, "gear": 3, "carb": 3},
    {"name": "Merc 450SLC", "mpg": 15.2, "cyl": 8, "disp": 275.8, "hp": 180, "drat": 3.07, "wt": 3.78, "qsec": 18.0, "vs": 0, "am": 0, "gear": 3, "carb": 3},
    {"name": "Cadillac Fleetwood", "mpg": 10.4, "cyl": 8, "disp": 472.0, "hp": 205, "drat": 2.93, "wt": 5.25, "qsec": 17.98, "vs": 0, "am": 0, "gear": 3, "carb": 4},
    {"name": "Lincoln Continental", "mpg": 10.4, "cyl": 8, "disp": 460.0, "hp": 215, "drat": 3.0, "wt": 5.424, "qsec": 17.82, "vs": 0, "am": 0, "gear": 3, "carb": 4},
    {"name": "Chrysler Imperial", "mpg": 14.7, "cyl": 8, "disp": 440.0, "hp": 230, "drat": 3.23, "wt": 5.345, "qsec": 17.42, "vs": 0, "am": 0, "gear": 3, "carb": 4},
    {"name": "Fiat 128", "mpg": 32.4, "cyl": 4, "disp": 78.7, "hp": 66, "drat": 4.08, "wt": 2.2, "qsec": 19.47, "vs": 1, "am": 1, "gear": 4, "carb": 1},
    {"name": "Honda Civic", "mpg": 30.4, "cyl": 4, "disp": 75.7, "hp": 52, "drat": 4.93, "wt": 1.615, "qsec": 18.52, "vs": 1, "am": 1, "gear": 4, "carb": 2},
    {"name": "Toyota Corolla", "mpg": 33.9, "cyl": 4, "disp": 71.1, "hp": 65, "drat": 4.22, "wt": 1.835, "qsec": 19.9, "vs": 1, "am": 1, "gear": 4, "carb": 1},
    {"name": "Toyota Corona", "mpg": 21.5, "cyl": 4, "disp": 120.1, "hp": 97, "drat": 3.7, "wt": 2.465, "qsec": 20.01, "vs": 1, "am": 0, "gear": 3, "carb": 1},
    {"name": "Dodge Challenger", "mpg": 15.5, "cyl": 8, "disp": 318.0, "hp": 150, "drat": 2.76, "wt": 3.52, "qsec": 16.87, "vs": 0, "am": 0, "gear": 3, "carb": 2},
    {"name": "AMC Javelin", "mpg": 15.2, "cyl": 8, "disp": 304.0, "hp": 150, "drat": 3.15, "wt": 3.435, "qsec": 17.3, "vs": 0, "am": 0, "gear": 3, "carb": 2},
    {"name": "Camaro Z28", "mpg": 13.3, "cyl": 8, "disp": 350.0, "hp": 245, "drat": 3.73, "wt": 3.84, "qsec": 15.41, "vs": 0, "am": 0, "gear": 3, "carb": 4},
    {"name": "Pontiac Firebird", "mpg": 19.2, "cyl": 8, "disp": 400.0, "hp": 175, "drat": 3.08, "wt": 3.845, "qsec": 17.05, "vs": 0, "am": 0, "gear": 3, "carb": 2},
    {"name": "Fiat X1-9", "mpg": 27.3, "cyl": 4, "disp": 79.0, "hp": 66, "drat": 4.08, "wt": 1.935, "qsec": 18.9, "vs": 1, "am": 1, "gear": 4, "carb": 1},
    {"name": "Porsche 914-2", "mpg": 26.0, "cyl": 4, "disp": 120.3, "hp": 91, "drat": 4.43, "wt": 2.14, "qsec": 16.7, "vs": 0, "am": 1, "gear": 5, "carb": 2},
    {"name": "Lotus Europa", "mpg": 30.4, "cyl": 4, "disp": 95.1, "hp": 113, "drat": 3.77, "wt": 1.513, "qsec": 16.9, "vs": 1, "am": 1, "gear": 5, "carb": 2},
    {"name": "Ford Pantera L", "mpg": 15.8, "cyl": 8, "disp": 351.0, "hp": 264, "drat": 4.22, "wt": 3.17, "qsec": 14.5, "vs": 0, "am": 1, "gear": 5, "carb": 4},
    {"name": "Ferrari Dino", "mpg": 19.7, "cyl": 6, "disp": 145.0, "hp": 175, "drat": 3.62, "wt": 2.77, "qsec": 15.5, "vs": 0, "am": 1, "gear": 5, "carb": 6},
    {"name": "Maserati Bora", "mpg": 15.0, "cyl": 8, "disp": 301.0, "hp": 335, "drat": 3.54, "wt": 3.57, "qsec": 14.6, "vs": 0, "am": 1, "gear": 5, "carb": 8},
    {"name": "Volvo 142E", "mpg": 21.4, "cyl": 4, "disp": 121.0, "hp": 109, "drat": 4.11, "wt": 2.78, "qsec": 18.6, "vs": 1, "am": 1, "gear": 4, "carb": 2}
]

FEATURES = ["cyl", "disp", "hp", "drat", "wt", "qsec", "vs", "am", "gear", "carb"]
TARGET = "mpg"

DEFAULT_HYPERPARAMS = {
    "fit_intercept": True,
    "ridge_penalty": 1e-6,
    "solver": "normal_equation",
    "target_rmse": 2.0,
    "target_mae": 1.5,
}

RL_ENVIRONMENTS = {
    "adaptive": {
        "strategy": "adaptive",
        "opt_metric": "rmse",
        "target": 2.5,
        "growth": 2.0,
        "decay": 1.5,
    },
    "explore_exploit": {
        "strategy": "explore_exploit",
        "opt_metric": "rmse",
        "candidate_penalties": [1e-6, 5e-6, 1e-5, 5e-5, 1e-4, 5e-4],
        "explore_probability": 0.5,
        "seed": 42,
    },
}

EVALUATION_KB = {
    "focus": [
        "Tie R-squared and adjusted R-squared back to generalisation confidence.",
        "Inspect RMSE versus residual standard error to gauge noise floor.",
        "Use variance inflation factors (VIF) to surface multicollinearity risks.",
    ],
    "reporting_style": "Be crisp, flag risks, and call out strengths.",
}

RECOMMENDATION_KB = {
    "multicollinearity_guidelines": [
        "VIF between 5 and 10 suggests strong correlation; consider dropping or transforming predictors.",
        "Regularisation (ridge or lasso) stabilises coefficients when predictors co-move.",
        "Centred or standardised features help compare magnitudes across scales.",
    ],
    "constraint_tips": [
        "Respect physical realism when adjusting displacement or weight.",
        "Remember `vs` and `am` encode engine and transmission categories.",
    ],
    "sign_expectations": [
        "Expect negative coefficients for load and power variables (`wt`, `cyl`, `disp`, `hp`) because higher values reduce fuel efficiency.",
        "Expect positive coefficients for efficiency indicators (`drat`, `am`, `gear`) because performance gearing and drivetrain efficiency support mpg.",
        "Reversed signs typically stem from multicollinearity, omitted interactions, or scaling issues - diagnose these drivers before acting.",
    ],
    "modeling_playbook": [
        "Test interaction terms (e.g., weight × transmission) if diagnostics show structured residuals.",
        "Stepwise feature pruning helps when high-VIF predictors dilute signal.",
        "If heavier cars show systematic error, try log-transforms on displacement or weight.",
    ],
}


def build_context_capability() -> ContextCapability:
    return ContextCapability(
        data={
            "dataset": {"data": MTCARS_DATA, "features": FEATURES, "target": TARGET},
            "hyperparameters": DEFAULT_HYPERPARAMS.copy(),
        }
    )


def build_message_bus() -> tuple[MessageBus, MessageBusCapability]:
    bus = MessageBus()
    return bus, MessageBusCapability(bus=bus)


def build_evaluation_resource() -> ResourceCapability:
    return ResourceCapability(name="knowledge.evaluation", resource=EVALUATION_KB, target="components")


def build_recommendation_resource() -> ResourceCapability:
    return ResourceCapability(name="knowledge.recommendation", resource=RECOMMENDATION_KB, target="components")


OPENAI_MODEL = "gpt-4o-mini"
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ") 
try:
    candidate_llm = EnvOpenAIAdapter(model=OPENAI_MODEL)
    if getattr(candidate_llm, "_client", None) is None:
        raise RuntimeError("OpenAI backend unavailable")
    llm_adapter = candidate_llm
except Exception:
    def _fallback(prompt: str) -> str:
        lines = [line.strip() for line in prompt.splitlines() if line.strip()]
        excerpt = lines[-1][:160] if lines else "mtcars regression analysis pending"
        return f"[stub llm] {excerpt}"

    llm_adapter = FunctionAdapter(_fallback)


In [2]:
# Cell 2: helper routines and modelling agent definition

import random
from math import sqrt
from typing import Any, Mapping, Sequence


class RidgePenaltyTrainer:
    """Simple tuner for ridge regression penalty supporting adaptive and explore/exploit modes."""

    def __init__(self, *, min_penalty: float = 1e-8, max_penalty: float = 1.0, random_seed: int | None = None) -> None:
        self.min_penalty = min_penalty
        self.max_penalty = max_penalty
        self._rng = random.Random(random_seed)

    def setup(self, context: Context, *, environment: Mapping[str, Any] | None = None) -> None:
        context.scratch.setdefault("rl_history", [])
        context.scratch.setdefault("model_history", [])
        if environment and "seed" in environment:
            self._rng.seed(int(environment["seed"]))

    def update(self, context: Context, step: AgentStep, *, environment: Mapping[str, Any] | None = None) -> None:
        env = dict(environment or {})
        strategy = env.get("strategy", "adaptive")
        metric_name = env.get("opt_metric", "rmse")

        metrics = context.scratch.get("model_outputs", {}).get("metrics", {})
        current_value = float(metrics.get(metric_name, 0.0))
        hyperparams = context.scratch["context"]["hyperparameters"]
        current_penalty = float(hyperparams.get("ridge_penalty", 1e-6))

        reward = 0.0
        action = "noop"
        target_value: float | None = None

        if strategy == "explore_exploit":
            state = context.scratch.setdefault(
                "rl_explore_state",
                {"best_metric": float("inf"), "best_penalty": current_penalty},
            )
            if current_value > 0 and current_value < state["best_metric"]:
                state["best_metric"] = current_value
                state["best_penalty"] = current_penalty

            candidates = [float(x) for x in env.get("candidate_penalties", [1e-6, 5e-6, 1e-5, 5e-5, 1e-4])]
            if not candidates:
                candidates = [current_penalty]
            explore_prob = float(env.get("explore_probability", 0.3))
            if state["best_metric"] == float("inf") or self._rng.random() < explore_prob:
                new_penalty = self._rng.choice(candidates)
                action = "explore"
            else:
                new_penalty = state["best_penalty"]
                action = "exploit"
            reward = state["best_metric"] - current_value if state["best_metric"] != float("inf") else 0.0
        else:
            target_value = float(env.get("target", current_value))
            growth = float(env.get("growth", 2.0))
            decay = float(env.get("decay", 1.5))
            if current_value > target_value:
                new_penalty = current_penalty * growth
                action = "increase"
            else:
                new_penalty = current_penalty / decay
                action = "decrease"
            reward = target_value - current_value

        new_penalty = min(self.max_penalty, max(self.min_penalty, new_penalty))
        hyperparams["ridge_penalty"] = new_penalty

        entry = {
            "iteration": context.scratch.get("iteration", 0),
            "strategy": strategy,
            "metric": metric_name,
            "value": round(current_value, 4),
            "action": action,
            "ridge_penalty": round(new_penalty, 6),
            "reward": round(reward, 4),
        }
        if strategy == "explore_exploit":
            explore_state = context.scratch.get("rl_explore_state", {})
            best_metric = explore_state.get("best_metric", float("inf"))
            entry["best_metric"] = round(best_metric, 4) if best_metric != float("inf") else None
        if target_value is not None:
            entry["target"] = round(target_value, 4)

        context.scratch.setdefault("rl_history", []).append(entry)

        try:
            bus = context.get_message_bus()
            bus.publish(
                agent="rl_trainer",
                content=f"[iter {entry['iteration']}] {strategy} -> {action} ridge_penalty={new_penalty:.6f}",
                iteration=context.scratch.get("iteration", 0),
                channel="rl",
            )
        except KeyError:
            pass


def build_design_row(row: Mapping[str, float], features: Sequence[str]) -> list[float]:
    return [1.0] + [float(row[f]) for f in features]


def solve_linear_system(matrix: Sequence[Sequence[float]], vector: Sequence[float]) -> list[float]:
    size = len(matrix)
    augmented = [list(matrix[i]) + [float(vector[i])] for i in range(size)]
    for col in range(size):
        pivot_row = max(range(col, size), key=lambda r: abs(augmented[r][col]))
        pivot_val = augmented[pivot_row][col]
        if abs(pivot_val) < 1e-12:
            continue
        if pivot_row != col:
            augmented[col], augmented[pivot_row] = augmented[pivot_row], augmented[col]
        pivot_val = augmented[col][col]
        for j in range(col, size + 1):
            augmented[col][j] /= pivot_val
        for row in range(size):
            if row == col:
                continue
            factor = augmented[row][col]
            if abs(factor) < 1e-12:
                continue
            for j in range(col, size + 1):
                augmented[row][j] -= factor * augmented[col][j]
    return [augmented[i][size] for i in range(size)]


def fit_linear_model(rows: Sequence[Mapping[str, float]], features: Sequence[str], target: str, ridge_penalty: float = 1e-6) -> dict:
    design = [build_design_row(row, features) for row in rows]
    y_values = [float(row[target]) for row in rows]
    cols = len(features) + 1
    xtx = [[0.0 for _ in range(cols)] for _ in range(cols)]
    xty = [0.0 for _ in range(cols)]
    for x_row, y_val in zip(design, y_values):
        for i in range(cols):
            xty[i] += x_row[i] * y_val
            for j in range(cols):
                xtx[i][j] += x_row[i] * x_row[j]
    for i in range(cols):
        xtx[i][i] += ridge_penalty
    coefficients = solve_linear_system(xtx, xty)
    predictions: list[float] = []
    residuals: list[float] = []
    for x_row, y_val in zip(design, y_values):
        prediction = sum(coefficients[idx] * x_row[idx] for idx in range(cols))
        predictions.append(prediction)
        residuals.append(y_val - prediction)
    n = len(rows)
    ss_res = sum(r ** 2 for r in residuals)
    mean_y = sum(y_values) / n
    ss_tot = sum((y_val - mean_y) ** 2 for y_val in y_values)
    r2 = 1.0 - ss_res / ss_tot if ss_tot else 0.0
    adj_r2 = 1.0 - (1.0 - r2) * (n - 1) / (n - len(features) - 1) if n > len(features) + 1 else float("nan")
    rmse = sqrt(ss_res / n)
    mae = sum(abs(r) for r in residuals) / n
    rse = sqrt(ss_res / (n - len(features) - 1)) if n > len(features) + 1 else float("nan")
    max_abs_residual = max(abs(r) for r in residuals)
    coeff_map = {"intercept": coefficients[0]}
    for index, feature in enumerate(features):
        coeff_map[feature] = coefficients[index + 1]
    metrics = {
        "r2": r2,
        "adjusted_r2": adj_r2,
        "rmse": rmse,
        "mae": mae,
        "residual_std_error": rse,
        "max_abs_residual": max_abs_residual,
    }
    return {
        "coefficients": coeff_map,
        "predictions": predictions,
        "residuals": residuals,
        "metrics": metrics,
    }


def compute_vif(rows: Sequence[Mapping[str, float]], features: Sequence[str], ridge_penalty: float) -> dict[str, float]:
    vifs: dict[str, float] = {}
    for feature in features:
        others = [fld for fld in features if fld != feature]
        if not others:
            vifs[feature] = 1.0
            continue
        transformed = []
        for row in rows:
            entry = {fld: row[fld] for fld in others}
            entry["_target"] = row[feature]
            transformed.append(entry)
        aux_model = fit_linear_model(transformed, others, "_target", ridge_penalty=ridge_penalty)
        r2 = aux_model["metrics"]["r2"]
        vifs[feature] = float("inf") if r2 >= 0.999999 else 1.0 / (1.0 - r2)
    return vifs


def modelling_step(ctx: Context) -> AgentStep:
    ctx.scratch.setdefault("iteration", 0)
    dataset = ctx.scratch["context"]["dataset"]
    hyperparams = ctx.scratch["context"]["hyperparameters"]
    rows = dataset["data"]
    features = dataset["features"]
    target = dataset["target"]
    model = fit_linear_model(rows, features, target, ridge_penalty=hyperparams.get("ridge_penalty", 1e-6))
    model["hyperparameters"] = dict(hyperparams)
    ctx.scratch["model_outputs"] = model
    ctx.scratch.setdefault("model_history", []).append(
        {
            "iteration": ctx.scratch.get("iteration", 0),
            "hyperparameters": dict(hyperparams),
            "metrics": model["metrics"],
        }
    )
    bus = ctx.get_message_bus()
    iteration = ctx.scratch.get("iteration", 0)
    coeff_summary = ", ".join(f"{name}={value:.3f}" for name, value in model["coefficients"].items())
    summary = (
        f"iteration={iteration} ridge_penalty={hyperparams.get('ridge_penalty', 0.0):.6f} | "
        f"coefficients: {coeff_summary}"
    )
    bus.publish(agent="modeling_agent", content=summary, iteration=iteration, channel="model")
    return AgentStep(
        out_messages=[Message(role="modeling_agent", content=summary)],
        state_updates={"model_outputs": model},
    )


In [3]:
# Cell 3: evaluation and recommendation agent steps


def evaluation_step(ctx: Context) -> AgentStep:
    dataset = ctx.scratch["context"]["dataset"]
    hyperparams = ctx.scratch["context"]["hyperparameters"]
    model_outputs = ctx.scratch.get("model_outputs")
    if not model_outputs:
        raise RuntimeError("Model outputs missing; run the modelling agent first.")
    rows = dataset["data"]
    features = dataset["features"]
    vif_scores = compute_vif(rows, features, ridge_penalty=hyperparams.get("ridge_penalty", 1e-6))
    metrics = dict(model_outputs["metrics"])
    metrics.update({
        "n_observations": len(rows),
        "mean_mpg": sum(float(row[dataset["target"]]) for row in rows) / len(rows),
    })
    vif_summary = {name: (float("inf") if value == float("inf") else round(value, 3)) for name, value in vif_scores.items()}
    diagnostics = {"metrics": metrics, "vif": vif_summary}
    latest_rl = ctx.scratch.get("rl_history", [])[-1] if ctx.scratch.get("rl_history") else None
    diagnostics["hyperparameters"] = dict(hyperparams)
    diagnostics["rl_adjustment"] = latest_rl
    knowledge = ctx.get_component("knowledge.evaluation")
    prompt_lines = [
        "You are the evaluation agent reviewing diagnostics for a linear regression on the mtcars dataset.",
        f"Focus reminders: {knowledge['focus']}",
        f"Key metrics: {diagnostics['metrics']}",
        f"Variance inflation factors: {diagnostics['vif']}",
        f"Current hyperparameters: {diagnostics['hyperparameters']}",
        "Flag the main insights in three concise sentences, noting strengths and risks.",
    ]
    if latest_rl:
        prompt_lines.append(f"Latest reinforcement learning adjustment: {latest_rl}")
    insights = ctx.llm.generate("".join(prompt_lines))
    diagnostics["insights"] = insights
    ctx.scratch["evaluation"] = diagnostics
    bus = ctx.get_message_bus()
    bus.publish(agent="evaluation_agent", content=insights, iteration=ctx.scratch.get("iteration", 0), channel="diagnostics")
    return AgentStep(
        out_messages=[Message(role="evaluation_agent", content=insights)],
        state_updates={"evaluation": diagnostics},
    )


def recommendation_step(ctx: Context) -> AgentStep:
    model_outputs = ctx.scratch.get("model_outputs")
    evaluation = ctx.scratch.get("evaluation")
    if not model_outputs or not evaluation:
        raise RuntimeError("Run modelling and evaluation agents before generating recommendations.")
    knowledge = ctx.get_component("knowledge.recommendation")
    coeff_view = {name: round(value, 4) for name, value in model_outputs["coefficients"].items()}
    rl_history = ctx.scratch.get("rl_history", [])
    latest_rl = rl_history[-1] if rl_history else None
    prompt_lines = [
        "You are the recommendation agent with deep linear regression expertise.",
        f"Knowledge base: {knowledge}",
        f"Hyperparameters: {model_outputs['hyperparameters']}",
        f"Coefficients: {coeff_view}",
        f"Diagnostics: metrics={evaluation['metrics']}, vif={evaluation['vif']}",
        f"Evaluation narrative: {evaluation['insights']}",
        "Investigate the underlying reasons for any issues or surprising signals and describe concrete fixes.",
        "Respond with a numbered list of three targeted recommendations covering regularisation, feature handling, and diagnostic follow-up, each with root-cause context and steps to resolve issues.",
    ]
    if latest_rl:
        prompt_lines.append(f"Latest reinforcement learning adjustment: {latest_rl}")
    guidance = ctx.llm.generate("".join(prompt_lines))
    ctx.scratch["recommendations"] = guidance
    history_entry = {
        "iteration": ctx.scratch.get("iteration", 0),
        "hyperparameters": dict(model_outputs["hyperparameters"]),
        "recommendation": guidance,
    }
    ctx.scratch.setdefault("recommendation_history", []).append(history_entry)
    bus = ctx.get_message_bus()
    bus.publish(agent="recommendation_agent", content=guidance, iteration=ctx.scratch.get("iteration", 0), channel="recommendations")
    return AgentStep(
        out_messages=[Message(role="recommendation_agent", content=guidance)],
        state_updates={"recommendations": guidance},
    )



def summary_step(ctx: Context) -> AgentStep:
    recommendation_history = ctx.scratch.get("recommendation_history", [])
    model_history = ctx.scratch.get("model_history", [])
    rl_history = ctx.scratch.get("rl_history", [])
    evaluation = ctx.scratch.get("evaluation", {})
    latest_metrics = ctx.scratch.get("model_outputs", {}).get("metrics", {})
    prompt_lines = [
        "You are the chief analyst summarising the mtcars regression agent run.",
        f"Latest metrics: {latest_metrics}",
        f"Recent model iterations: {model_history[-3:] if model_history else []}",
        f"Recent reinforcement learning adjustments: {rl_history[-3:] if rl_history else []}",
        f"Evaluation summary: {evaluation.get('insights', 'n/a')}",
        f"Recent recommendations: {[entry['recommendation'][:160] for entry in recommendation_history[-3:]] if recommendation_history else []}",
        "Explain why the selected model is reliable, citing concrete evidence, and justify how the recommendations align with the diagnostics.",
        "Organise the response with a short overview, evidence-backed bullet list, and a closing next-step paragraph.",
    ]
    explanation = ctx.llm.generate("".join(prompt_lines)) if ctx.llm else "Summary unavailable: no LLM adapter."
    summary_payload = {
        "explanation": explanation,
        "evidence": {
            "metrics": latest_metrics,
            "recent_model_iterations": model_history[-3:] if model_history else [],
            "recent_rl_steps": rl_history[-3:] if rl_history else [],
            "recent_recommendations": recommendation_history[-3:] if recommendation_history else [],
        },
    }
    ctx.scratch["final_summary"] = summary_payload
    try:
        bus = ctx.get_message_bus()
        bus.publish(
            agent="summary_agent",
            content=explanation,
            iteration=ctx.scratch.get("iteration", 0),
            channel="summary",
        )
    except KeyError:
        pass
    return AgentStep(
        out_messages=[Message(role="summary_agent", content=explanation)],
        state_updates={"final_summary": summary_payload},
    )


In [4]:
# Cell 3b: JEPA alignment helpers

def jepa_predictor(ctx: Context) -> tuple[float, float]:
    metrics = ctx.scratch.get("model_outputs", {}).get("metrics", {})
    return (
        float(metrics.get("rmse", 0.0)),
        float(metrics.get("mae", 0.0)),
    )


def jepa_target(ctx: Context) -> tuple[float, float]:
    evaluation = ctx.scratch.get("evaluation", {})
    metrics = evaluation.get("metrics", {})
    target_rmse = float(ctx.scratch.get("context", {}).get("hyperparameters", {}).get("target_rmse", 2.0))
    target_mae = float(ctx.scratch.get("context", {}).get("hyperparameters", {}).get("target_mae", 1.5))
    return (
        float(metrics.get("rmse", target_rmse)),
        float(metrics.get("mae", target_mae)),
    )


def jepa_loss(predictor_vec: tuple[float, float], target_vec: tuple[float, float]) -> float:
    return float(sum(abs(p - t) for p, t in zip(predictor_vec, target_vec)))


def jepa_update(
    ctx: Context,
    predictor_vec: tuple[float, float],
    target_vec: tuple[float, float],
    loss: float,
) -> None:
    ctx.scratch.setdefault("jepa_adjustments", []).append(
        {
            "iteration": ctx.scratch.get("iteration", 0),
            "predictor": predictor_vec,
            "target": target_vec,
            "loss": loss,
        }
    )
    hyperparams = ctx.scratch.get("context", {}).get("hyperparameters", {})
    ridge_penalty = float(hyperparams.get("ridge_penalty", 1e-6))
    if loss > 0.5:
        ridge_penalty = min(1e-2, ridge_penalty * 1.1)
    else:
        ridge_penalty = max(1e-8, ridge_penalty * 0.95)
    hyperparams["ridge_penalty"] = ridge_penalty


In [5]:
# Cell 4: assemble agents and run the multi-strategy pipeline

from typing import Any, Mapping


def run_pipeline(strategy: str, rl_environment: Mapping[str, Any], iterations: int = 3) -> Mapping[str, Any]:
    bus, bus_capability = build_message_bus()
    context_capability = build_context_capability()
    evaluation_resource = build_evaluation_resource()
    recommendation_resource = build_recommendation_resource()

    trainer = RidgePenaltyTrainer(min_penalty=1e-7, max_penalty=0.5, random_seed=rl_environment.get("seed"))

    modeling_agent = (
        AgentBuilder(name="modeling_agent", role="modeler")
        .with_capabilities([bus_capability, context_capability])
        .with_reinforcement_learning(trainer, environment=rl_environment)
        .with_jepa_learning(
            predictor=jepa_predictor,
            target=jepa_target,
            loss_fn=jepa_loss,
            update_fn=jepa_update,
        )
        .with_step(modelling_step)
        .build()
    )

    evaluation_agent = (
        AgentBuilder(name="evaluation_agent", role="quality")
        .with_capabilities([bus_capability, context_capability, evaluation_resource])
        .with_llm(llm_adapter)
        .with_step(evaluation_step)
        .build()
    )

    recommendation_agent = (
        AgentBuilder(name="recommendation_agent", role="advisor")
        .with_capabilities([bus_capability, context_capability, recommendation_resource])
        .with_llm(llm_adapter)
        .with_step(recommendation_step)
        .build()
    )

    summary_agent = (
        AgentBuilder(name="summary_agent", role="analyst")
        .with_capabilities([bus_capability])
        .with_llm(llm_adapter)
        .with_step(summary_step)
        .build()
    )

    context = Context(goal=f"Diagnose linear regression for mtcars mpg ({strategy})")
    context.add_component("message_bus", bus)

    iteration_summaries: list[dict[str, float]] = []

    for iteration in range(iterations):
        context.scratch["iteration"] = iteration
        for agent in (modeling_agent, evaluation_agent, recommendation_agent):
            result = agent.run(context)
            if result.out_messages:
                context.push_message(result.out_messages[-1])
        iteration_summaries.append(
            {
                "iteration": iteration,
                "ridge_penalty": float(context.scratch["context"]["hyperparameters"]["ridge_penalty"]),
                "rmse": float(context.scratch["model_outputs"]["metrics"]["rmse"]),
            }
        )

    summary_result = summary_agent.run(context)
    if summary_result.out_messages:
        context.push_message(summary_result.out_messages[-1])

    final_summary = context.scratch.get("final_summary", {})

    return {
        "strategy": strategy,
        "iterations": iteration_summaries,
        "model_metrics": context.scratch["model_outputs"]["metrics"],
        "vif": context.scratch["evaluation"]["vif"],
        "recommendation_history": context.scratch.get("recommendation_history", []),
        "rl_history": context.scratch.get("rl_history", []),
        "model_history": context.scratch.get("model_history", []),
        "final_hyperparameters": dict(context.scratch["context"]["hyperparameters"]),
        "final_summary": final_summary,
        "message_bus": [
            {"agent": record.agent, "channel": record.channel, "message": record.content}
            for record in bus.conversation()
        ],
        "latest_recommendation": context.scratch.get("recommendations", ""),
        "summary_message": summary_result.out_messages[-1].content if summary_result.out_messages else "",
        "jepa_history": context.scratch.get("jepa_history", []),
    }


pipeline_reports = {
    strategy: run_pipeline(strategy, env, iterations=3)
    for strategy, env in RL_ENVIRONMENTS.items()
}

pipeline_reports


{'adaptive': {'strategy': 'adaptive',
  'iterations': [{'iteration': 0,
    'ridge_penalty': 6.333333333333332e-07,
    'rmse': 2.14690496721448},
   {'iteration': 1,
    'ridge_penalty': 4.01111111111111e-07,
    'rmse': 2.1469049671824187},
   {'iteration': 2,
    'ridge_penalty': 2.5403703703703695e-07,
    'rmse': 2.1469049671695575}],
  'model_metrics': {'r2': 0.8690157644767136,
   'adjusted_r2': 0.8066423189894343,
   'rmse': 2.1469049671695575,
   'mae': 1.722740155112209,
   'residual_std_error': 2.6501970278761413,
   'max_abs_residual': 4.627091046750344},
  'vif': {'cyl': 15.374,
   'disp': 21.62,
   'hp': 9.832,
   'drat': 3.375,
   'wt': 15.165,
   'qsec': 7.528,
   'vs': 4.966,
   'am': 4.648,
   'gear': 5.357,
   'carb': 7.909},
  'recommendation_history': [{'iteration': 0,
    'hyperparameters': {'fit_intercept': True,
     'ridge_penalty': 1e-06,
     'solver': 'normal_equation',
     'target_rmse': 2.0,
     'target_mae': 1.5},
    'recommendation': "Based on the eva

In [6]:
# Cell 5: inspect message bus conversation snapshots

{
    strategy: [
        {
            "agent": record["agent"],
            "channel": record["channel"],
            "message": record["message"][:160],
        }
        for record in report["message_bus"]
    ]
    for strategy, report in pipeline_reports.items()
}


{'adaptive': [{'agent': 'modeling_agent',
   'channel': 'model',
   'message': 'iteration=0 ridge_penalty=0.000001 | coefficients: intercept=12.303, cyl=-0.111, disp=0.013, hp=-0.021, drat=0.787, wt=-3.715, qsec=0.821, vs=0.318, am=2.520, g'},
  {'agent': 'rl_trainer',
   'channel': 'rl',
   'message': '[iter 0] adaptive -> decrease ridge_penalty=0.000001'},
  {'agent': 'evaluation_agent',
   'channel': 'diagnostics',
   'message': 'The linear regression model demonstrates a strong fit with an R-squared of 0.87 and an adjusted R-squared of 0.81, indicating good generalization confidence; ho'},
  {'agent': 'recommendation_agent',
   'channel': 'recommendations',
   'message': 'Based on the evaluation of your linear regression model, here are three targeted recommendations to address the issues identified, particularly focusing on mult'},
  {'agent': 'modeling_agent',
   'channel': 'model',
   'message': 'iteration=1 ridge_penalty=0.000001 | coefficients: intercept=12.303, cyl=-0.111, di

In [7]:
# Cell 6: recommendation history per strategy

{
    strategy: [
        {
            "iteration": entry["iteration"],
            "hyperparameters": entry["hyperparameters"],
            "excerpt": entry["recommendation"][:160],
        }
        for entry in report.get("recommendation_history", [])
    ]
    for strategy, report in pipeline_reports.items()
}


{'adaptive': [{'iteration': 0,
   'hyperparameters': {'fit_intercept': True,
    'ridge_penalty': 1e-06,
    'solver': 'normal_equation',
    'target_rmse': 2.0,
    'target_mae': 1.5},
   'excerpt': 'Based on the evaluation of your linear regression model, here are three targeted recommendations to address the issues identified, particularly focusing on mult'},
  {'iteration': 1,
   'hyperparameters': {'fit_intercept': True,
    'ridge_penalty': 6.333333333333332e-07,
    'solver': 'normal_equation',
    'target_rmse': 2.0,
    'target_mae': 1.5},
   'excerpt': 'Based on the evaluation of the linear regression model and the diagnostics provided, here are three targeted recommendations to address the identified issues, p'},
  {'iteration': 2,
   'hyperparameters': {'fit_intercept': True,
    'ridge_penalty': 4.01111111111111e-07,
    'solver': 'normal_equation',
    'target_rmse': 2.0,
    'target_mae': 1.5},
   'excerpt': 'Based on the evaluation of your linear regression model, here 

In [8]:
# Cell 7: strategy summaries with evidence

{
    strategy: {
        "summary": report.get("final_summary", {}).get("explanation", ""),
        "evidence": report.get("final_summary", {}).get("evidence", {}),
    }
    for strategy, report in pipeline_reports.items()
}


{'adaptive': {'summary': "### Overview\nThe linear regression model applied to the mtcars dataset has demonstrated strong predictive capabilities, as evidenced by its performance metrics. While the model shows promise, there are areas for improvement, particularly concerning multicollinearity among predictors. The following analysis provides concrete evidence of the model's reliability and outlines targeted recommendations to enhance its performance.\n\n### Evidence-Backed Metrics\n- **R-squared Value**: The model achieved an R-squared of approximately **0.87**, indicating that about **87%** of the variance in the target variable can be explained by the predictors. This high value suggests a strong fit.\n- **Adjusted R-squared**: The adjusted R-squared of **0.81** accounts for the number of predictors in the model, reinforcing the model's generalization capability and indicating that it is not overfitting.\n- **Root Mean Square Error (RMSE)**: The RMSE of **2.15** is close to the targe

In [9]:
# Cell 8: top regression models across strategies

from typing import List, Dict, Mapping

def collect_top_models(reports: Mapping[str, Mapping[str, Any]], top_n: int = 3) -> List[Dict[str, Any]]:
    candidates: List[Dict[str, Any]] = []
    for strategy, report in reports.items():
        model_history = {entry["iteration"]: entry for entry in report.get("model_history", [])}
        rl_history = {entry["iteration"]: entry for entry in report.get("rl_history", [])}
        recommendations = {entry["iteration"]: entry for entry in report.get("recommendation_history", [])}
        for iteration, record in model_history.items():
            metrics = record.get("metrics", {})
            rmse = metrics.get("rmse") or metrics.get("rmse", float("inf"))
            reward = rl_history.get(iteration, {}).get("reward", 0.0)
            composite_score = (1.0 / (1.0 + rmse)) + reward if rmse not in (None, float("inf")) else reward
            recommendation_excerpt = recommendations.get(iteration, {}).get("recommendation", "")[:200]
            explanation = (
                f"Strategy {strategy} iteration {iteration} achieved RMSE {rmse:.3f} "
                f"with reward {reward:.3f}. Recommendation focus: {recommendation_excerpt}"
            )
            candidates.append(
                {
                    "strategy": strategy,
                    "iteration": iteration,
                    "composite_score": round(composite_score, 4),
                    "metrics": metrics,
                    "hyperparameters": record.get("hyperparameters", {}),
                    "reward": reward,
                    "recommendation_excerpt": recommendation_excerpt,
                    "explanation": explanation,
                }
            )
    return sorted(candidates, key=lambda item: item["composite_score"], reverse=True)[:top_n]

TOP_REGRESSION_MODELS = collect_top_models(pipeline_reports, top_n=3)
TOP_REGRESSION_MODELS


[{'strategy': 'adaptive',
  'iteration': 0,
  'composite_score': 0.6709,
  'metrics': {'r2': 0.8690157644712321,
   'adjusted_r2': 0.8066423189813425,
   'rmse': 2.14690496721448,
   'mae': 1.7227401434992555,
   'residual_std_error': 2.650197027931595,
   'max_abs_residual': 4.627086350564831},
  'hyperparameters': {'fit_intercept': True,
   'ridge_penalty': 1e-06,
   'solver': 'normal_equation',
   'target_rmse': 2.0,
   'target_mae': 1.5},
  'reward': 0.3531,
  'recommendation_excerpt': 'Based on the evaluation of your linear regression model, here are three targeted recommendations to address the issues identified, particularly focusing on multicollinearity, feature handling, and dia',
  'explanation': 'Strategy adaptive iteration 0 achieved RMSE 2.147 with reward 0.353. Recommendation focus: Based on the evaluation of your linear regression model, here are three targeted recommendations to address the issues identified, particularly focusing on multicollinearity, feature handling,

In [10]:
# Cell 9: JEPA alignment log per strategy

{
    strategy: report.get("jepa_history", [])[-3:]
    for strategy, report in pipeline_reports.items()
}


{'adaptive': [{'predictor': (2.14690496721448, 1.7227401434992555),
   'target': (2.0, 1.5),
   'loss': 0.36964511071373574},
  {'predictor': (2.1469049671824187, 1.7227401506090014),
   'target': (2.14690496721448, 1.7227401434992555),
   'loss': 7.141807367716524e-09},
  {'predictor': (2.1469049671695575, 1.722740155112209),
   'target': (2.1469049671824187, 1.7227401506090014),
   'loss': 4.516068852211674e-09}],
 'explore_exploit': [{'predictor': (2.14690496721448, 1.7227401434992555),
   'target': (2.0, 1.5),
   'loss': 0.36964511071373574},
  {'predictor': (2.1469049672092604, 1.7227401444687236),
   'target': (2.14690496721448, 1.7227401434992555),
   'loss': 9.746878859573371e-10},
  {'predictor': (2.1469049719885125, 1.7227399788823903),
   'target': (2.1469049672092604, 1.7227401444687236),
   'loss': 1.7036558541683178e-07}]}